# v20.1 OmniQuant 2048-token preflight

Attach the frozen v20.1 bundle as a Kaggle dataset and enable one GPU. This notebook runs only the registered L0/L12/L23 smoke steps and the L0 8-step sequence. It does not run formal 20-epoch training or PPL. The exact Qwen revision will be downloaded if a locally mounted copy does not pass the frozen file hashes.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, zipfile

matches = [p for p in Path('/kaggle/input').rglob('configs/m3_preflight.json') if p.parent.parent.name == 'kaggle_bundle']
if not matches:
    archives = list(Path('/kaggle/input').rglob('kaggle_bundle_v20_1.zip'))
    if len(archives) != 1:
        raise RuntimeError(f'Expected one bundle or bundle zip under /kaggle/input; zip matches: {archives}')
    unpack = Path('/kaggle/working/v20_1_bundle_unpacked')
    unpack.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as zf:
        zf.extractall(unpack)
    matches = list(unpack.rglob('configs/m3_preflight.json'))
if len(matches) != 1:
    raise RuntimeError(f'Expected one mounted v20.1 bundle, found {matches}')
BUNDLE = matches[0].parent.parent
OUT = Path('/kaggle/working/range_w4_omniquant_kaggle_v20_1')
OUT.mkdir(parents=True, exist_ok=True)
print('bundle:', BUNDLE)
print('output:', OUT)
print('GPU selected by runner: cuda:0 only')

In [ ]:
# Pin only the Python packages used by the validated port; keep Kaggle's CUDA torch build.
import importlib.metadata as md
pins = {'transformers': '4.49.0', 'tokenizers': '0.21.4', 'accelerate': '1.14.0', 'safetensors': '0.8.0', 'psutil': '7.2.2'}
mismatch = []
for name, wanted in pins.items():
    try:
        got = md.version(name)
    except md.PackageNotFoundError:
        got = None
    if got != wanted:
        mismatch.append((name, got, wanted))
if mismatch:
    print('Installing frozen non-torch package pins:', mismatch)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(BUNDLE / 'requirements_kaggle.txt')], check=True)
env = subprocess.run([sys.executable, str(BUNDLE / 'environment_check.py'), '--output', str(OUT / 'KAGGLE_ENVIRONMENT.json'), '--disk-path', str(OUT)], check=False)
if env.returncode != 0:
    raise RuntimeError('CUDA or frozen package environment check failed')
verify = subprocess.run([sys.executable, str(BUNDLE / 'verify_bundle.py'), '--bundle-root', str(BUNDLE), '--output', str(OUT / 'BUNDLE_VERIFY.json')], check=False)
if verify.returncode != 0:
    raise RuntimeError('Bundle integrity failed')
subprocess.run([sys.executable, str(BUNDLE / 'audit_step_count.py'), '--bundle-root', str(BUNDLE), '--output', str(OUT / 'FORMAL_STEP_COUNT.json')], check=True)
subprocess.run([sys.executable, str(BUNDLE / 'audit_calibration.py'), '--manifest', str(BUNDLE / 'calibration/CALIBRATION_MANIFEST.json'), '--output', str(OUT / 'KAGGLE_CALIBRATION_CHECK.json')], check=True)

In [ ]:
# The runner verifies the exact model revision and every frozen model file hash before training steps.
model_candidates = [p.parent for p in Path('/kaggle/input').rglob('model.safetensors') if p.parent.name == '7ae557604adf67be50417f59c2c2f167def9a775' or (p.parent / 'MODEL_REVISION.json').is_file()]
cmd = [sys.executable, str(BUNDLE / 'run_preflight.py'), '--bundle-root', str(BUNDLE), '--output-dir', str(OUT)]
if len(model_candidates) == 1:
    cmd += ['--model-path', str(model_candidates[0])]
elif len(model_candidates) > 1:
    raise RuntimeError(f'Multiple candidate model snapshots found: {model_candidates}')
result = subprocess.run(cmd, check=False)
print('preflight process exit:', result.returncode)
if result.returncode not in (0, 1):
    raise RuntimeError('Preflight execution error; inspect the output directory')
verdict_path = OUT / 'PREFLIGHT_VERDICT.json'
if verdict_path.is_file():
    print(json.loads(verdict_path.read_text()))
else:
    print('No terminal verdict artifact was written; inspect traceback and partial outputs.')

In [ ]:
# Package results for download from Kaggle's notebook output panel.
import shutil
archive = shutil.make_archive('/kaggle/working/range_w4_omniquant_kaggle_v20_1', 'zip', root_dir=OUT)
print('result archive:', archive)
print('artifacts:', sorted(p.name for p in OUT.iterdir()))